# 문제 2 — 클로징 과제 잉여 인력 재배치
**입력 파일**: `R&D_Closing_DB_v2.xlsx`
- `소속과제명`이 비어있는 인원 → 재배치 대상 (잉여 인력)
- `소속과제명`이 있는 인원 → 기존 팀원 (고정)

**흐름**
```
1단계: 기존 팀 현황 EDA + 후보 과제 선택 + 과제별 추가 상한 설정
2단계: 잉여 인력 개인별 적합도 산출 (카드 1~4)
3단계: ILP 최적 재배치
```

## 0. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
print('라이브러리 로드 완료')

## 1. 데이터 로드 및 분리

In [ ]:
FILE = 'R&D_Closing_DB_v2.xlsx'

df_hr      = pd.read_excel(FILE, sheet_name='1. 인사DB')
df_skill   = pd.read_excel(FILE, sheet_name='2. R&D 요소기술 점수')
df_mapping = pd.read_excel(FILE, sheet_name='3. 과제별 요소역량 매핑')

# 스킬 컬럼
SKILL_COLS = [c for c in df_skill.columns if c.startswith('S')]
ALL_SKILLS = SKILL_COLS

# 과제-스킬 매핑 파싱
proj_req = {}
cur_proj = None
for _, row in df_mapping.iterrows():
    if pd.notna(row['과제코드']):
        cur_proj = row['과제코드']
        proj_req[cur_proj] = []
    if cur_proj and pd.notna(row['필수 요소기술 (주요 요건)']):
        proj_req[cur_proj].append(row['필수 요소기술 (주요 요건)'])

# 재배치 대상 / 기존 팀원 분리
surplus_hr    = df_hr[df_hr['소속과제명'].fillna('') == ''].reset_index(drop=True)
existing_hr   = df_hr[df_hr['소속과제명'].fillna('') != ''].reset_index(drop=True)
surplus_skill = df_skill[df_skill['소속과제명'].fillna('') == ''].reset_index(drop=True)
existing_skill= df_skill[df_skill['소속과제명'].fillna('') != ''].reset_index(drop=True)

ALL_PROJECTS   = sorted(existing_hr['소속과제명'].unique().tolist())
N_SURPLUS      = len(surplus_hr)
N_EXISTING     = len(existing_hr)

print(f'재배치 대상 (잉여 인력): {N_SURPLUS}명')
print(f'기존 팀원: {N_EXISTING}명')
print(f'후보 과제: {ALL_PROJECTS}')
print(f'스킬 수: {len(ALL_SKILLS)}개 | 과제 수: {len(proj_req)}개')

## 2. 전처리

In [ ]:
# ── 스킬 wide matrix ─────────────────────────────────────────────────────
# 잉여 인력 스킬 (사번 index, 스킬 컬럼, NaN→0)
surplus_wide = surplus_skill.set_index('사번')[SKILL_COLS].fillna(0)

# 기존 팀원 스킬
existing_wide = existing_skill.set_index('사번')[SKILL_COLS].fillna(0)

# ── 전체 인원 기준 스킬 메타 (IDF, 난이도용) ──────────────────────────────
all_wide = pd.concat([surplus_wide, existing_wide])
N_TOTAL  = len(all_wide)

holders_all   = (all_wide > 0).sum()       # 전체 기준 보유자 수
holders_lv4   = (all_wide >= 4).sum()      # 전체 기준 레벨4+ 보유자 수

# ── 잉여 인력 기준 스킬 메타 (KSS 분모용) ────────────────────────────────
holders_surplus = (surplus_wide > 0).sum()  # 잉여 인력 기준 보유자 수

# ── 팀별 정원 및 기존 팀원 스킬 기여분 ───────────────────────────────────
team_members = {}   # 과제 → 기존 팀원 사번 리스트
for p in ALL_PROJECTS:
    members = existing_hr[existing_hr['소속과제명'] == p]['사번'].tolist()
    team_members[p] = members

Nj_existing = {p: len(team_members[p]) for p in ALL_PROJECTS}  # 기존 팀원 수

# 과제별 기존 팀원 스킬 레벨 합 (소프트 제약 고정 기여분)
team_skill_sum = {}   # (과제, 스킬) → 기존팀원 레벨 합
team_skill_holders = {}  # (과제, 스킬) → 기존팀원 보유자 수 (보유커버리지 고정분)
for p in ALL_PROJECTS:
    mbr_wide = existing_wide.loc[existing_wide.index.isin(team_members[p])]
    for s in ALL_SKILLS:
        team_skill_sum[(p, s)]     = float(mbr_wide[s].sum()) if s in mbr_wide.columns else 0.0
        team_skill_holders[(p, s)] = int((mbr_wide[s] > 0).sum()) if s in mbr_wide.columns else 0

print('전처리 완료')
print(f'  전체 기준 보유자 수 (IDF용): {N_TOTAL}명')
print(f'  잉여 인력 기준 보유자 수 (KSS 분모용): {N_SURPLUS}명')
print(f'  과제별 기존 팀원 기여분 계산 완료')

## 1단계 — 기존 팀 현황 EDA

> 후보 과제 선택 및 과제별 최대 재배치 인원 설정

In [ ]:
# ── AVG_LEVEL 기준 설정 ──────────────────────────────────────────────────
AVG_LEVEL = 2.8   # 팀 스킬 레벨 합 기준 (3단계 소프트 제약과 동일하게 맞춤)

# ── 팀별 스킬 갭 계산 ────────────────────────────────────────────────────────
# 갭(p, s) = max(0, AVG_LEVEL × 팀정원 - 기존팀원 레벨합)
team_gap = {}      # (과제, 스킬) → 부족량
team_gap_count = {}  # 과제 → 미달 스킬 수
for p in ALL_PROJECTS:
    n_team = Nj_existing[p]
    gap_cnt = 0
    for s in proj_req.get(p, []):
        gap = max(0, AVG_LEVEL * n_team - team_skill_sum.get((p, s), 0))
        team_gap[(p, s)] = gap
        if gap > 0:
            gap_cnt += 1
    team_gap_count[p] = gap_cnt

# ── 팀별 성비 / 직급 ─────────────────────────────────────────────────────────
rmap = {'사원':1, '대리':2, '과장':3, '차장':4, '부장':5}
existing_hr['직급수'] = existing_hr['직위'].map(rmap)

team_stats = []
for p in ALL_PROJECTS:
    mbr = existing_hr[existing_hr['소속과제명'] == p]
    n   = len(mbr)
    n_f = (mbr['성별'] == '여').sum()
    avg_rk = mbr['직급수'].mean()
    req_skills = proj_req.get(p, [])
    n_gap_skills = team_gap_count[p]
    total_gap = sum(team_gap.get((p,s),0) for s in req_skills)
    team_stats.append({
        '과제': p, '현재인원': n,
        '여성수': int(n_f), '여성비율(%)': round(n_f/n*100 if n>0 else 0, 1),
        '평균직급': round(avg_rk, 2),
        '필수스킬수': len(req_skills),
        '미달스킬수': n_gap_skills,
        '총갭': round(total_gap, 1)
    })
team_stats_df = pd.DataFrame(team_stats)

print('[기존 팀 현황]')
display(team_stats_df)

In [ ]:
# ── 팀별 스킬 갭 히트맵 ────────────────────────────────────────────────────
all_req_skills = sorted(set(s for p in ALL_PROJECTS for s in proj_req.get(p, [])))
gap_matrix = pd.DataFrame(index=ALL_PROJECTS, columns=all_req_skills, dtype=float)
for p in ALL_PROJECTS:
    for s in all_req_skills:
        gap_matrix.loc[p, s] = team_gap.get((p, s), np.nan)

fig, ax = plt.subplots(figsize=(18, 8))
data = gap_matrix.fillna(0).values
im = ax.imshow(data, aspect='auto', cmap='Reds')
ax.set_xticks(range(len(all_req_skills))); ax.set_xticklabels(all_req_skills, rotation=90, fontsize=8)
ax.set_yticks(range(len(ALL_PROJECTS))); ax.set_yticklabels(ALL_PROJECTS, fontsize=9)
plt.colorbar(im, ax=ax, label='스킬 갭 (부족량)')
ax.set_title(f'팀별 스킬 갭 현황 (AVG_LEVEL={AVG_LEVEL} 기준)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 팀별 성비 / 직급 시각화 ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pool_f = (existing_hr['성별'] == '여').mean()
pool_r = existing_hr['직급수'].mean()

axes[0].bar(team_stats_df['과제'], team_stats_df['여성비율(%)'], color='#6366f1aa')
axes[0].axhline(pool_f*100, color='red', linestyle='--', alpha=0.7, label=f'전체 평균 {pool_f*100:.1f}%')
axes[0].set_title('팀별 여성 비율 (%)', fontsize=11); axes[0].legend(); axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(team_stats_df['과제'], team_stats_df['평균직급'], color='#f59e0baa')
axes[1].axhline(pool_r, color='red', linestyle='--', alpha=0.7, label=f'전체 평균 {pool_r:.2f}')
axes[1].set_title('팀별 평균 직급', fontsize=11); axes[1].legend(); axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 후보 과제 선택 — 재배치 인원을 받지 않을 팀 제외

In [ ]:
cb_exclude = {}
rows_exclude = []
for p in ALL_PROJECTS:
    row = team_stats_df[team_stats_df['과제']==p].iloc[0]
    label = (f"{p}  │  현재 {row['현재인원']}명  │  미달스킬 {row['미달스킬수']}개  "
             f"│  총갭 {row['총갭']}  │  여성 {row['여성비율(%)']}%  │  평균직급 {row['평균직급']}")
    cb = widgets.Checkbox(value=False, description=label,
                          layout=widgets.Layout(width='700px'),
                          style={'description_width':'initial'})
    cb_exclude[p] = cb
    rows_exclude.append(cb)

btn_clear_ex = widgets.Button(description='전체 해제', button_style='', layout=widgets.Layout(width='120px'))
def clear_exclude(_):
    for cb in cb_exclude.values(): cb.value = False
btn_clear_ex.on_click(clear_exclude)

header_ex = widgets.HTML('<b>재배치 인원을 받지 않을 팀을 선택하세요 (체크 = 제외)</b><br><br>')
display(widgets.VBox([header_ex, btn_clear_ex,
                      widgets.VBox(rows_exclude, layout=widgets.Layout(border='1px solid #ccc', padding='8px'))]))

### 과제별 최대 재배치 인원 설정

In [ ]:
# 공통 기본값 먼저 설정
default_max = widgets.BoundedIntText(value=5, min=0, max=50, description='과제당 기본 상한:', 
                                      style={'description_width':'initial'},
                                      layout=widgets.Layout(width='280px'))
apply_btn = widgets.Button(description='기본값 적용', button_style='primary', layout=widgets.Layout(width='120px'))

max_inputs = {}
max_rows = []
for p in ALL_PROJECTS:
    inp = widgets.BoundedIntText(value=5, min=0, max=50, description=f'{p} 최대:',
                                  style={'description_width':'initial'},
                                  layout=widgets.Layout(width='200px'))
    max_inputs[p] = inp
    max_rows.append(inp)

def apply_default(_):
    for inp in max_inputs.values():
        inp.value = default_max.value
apply_btn.on_click(apply_default)

header_max = widgets.HTML('<b>과제별 최대 재배치 인원 설정</b><br><br>')
default_row = widgets.HBox([default_max, apply_btn])
grid_rows = [widgets.HBox(max_rows[i:i+4]) for i in range(0, len(max_rows), 4)]
display(widgets.VBox([header_max, default_row, widgets.VBox(grid_rows,
                       layout=widgets.Layout(border='1px solid #ccc', padding='8px'))]))

## 2단계 — 잉여 인력 개인별 적합도 산출
> **위 1단계 설정 완료 후 이 셀부터 실행하세요.**

In [ ]:
# ── 후보 과제 확정 ───────────────────────────────────────────────────────
excluded    = [p for p, cb in cb_exclude.items() if cb.value]
CANDIDATE_P = [p for p in ALL_PROJECTS if p not in excluded]
MAX_ADD     = {p: max_inputs[p].value for p in CANDIDATE_P}

print(f'제외 과제: {excluded if excluded else "없음"}')
print(f'후보 과제 ({len(CANDIDATE_P)}개): {CANDIDATE_P}')
print(f'과제별 추가 상한: {MAX_ADD}')
print(f'재배치 대상: {N_SURPLUS}명')

In [ ]:
# ── IDF 계산 (전체 기준) ─────────────────────────────────────────────────
idf_raw  = np.log((N_TOTAL + 1) / (holders_all.reindex(ALL_SKILLS).fillna(0) + 1))
idf_min, idf_max = idf_raw.min(), idf_raw.max()
idf_norm = ((idf_raw - idf_min) / (idf_max - idf_min)).round(4)

df_idf = pd.DataFrame({
    '스킬': ALL_SKILLS,
    '전체보유자': holders_all.reindex(ALL_SKILLS).fillna(0).astype(int),
    'IDF_norm': idf_norm.reindex(ALL_SKILLS)
}).sort_values('IDF_norm', ascending=False).reset_index(drop=True)

q67 = df_idf['IDF_norm'].quantile(0.67); q33 = df_idf['IDF_norm'].quantile(0.33)
def idf_class(v):
    return '희귀' if v >= q67 else ('보편' if v <= q33 else '보통')
df_idf['분류'] = df_idf['IDF_norm'].apply(idf_class)

# ── KSS 계산 (문제 2 전용) ───────────────────────────────────────────────
# 분자: 기존팀원만으로 AVG_LEVEL 미달인 후보 과제 수
# 분모: 잉여 인력 중 보유자 수
# 4번 수정: 보유자 0명 + 수요 있음 → 외부 충원 필요 (KSS 최대값)
#           보유자 0명 + 수요 없음 → 무시 (KSS 0)
demand_gap = {}
for s in ALL_SKILLS:
    cnt = 0
    for p in CANDIDATE_P:
        if s in proj_req.get(p, []):
            if team_gap.get((p, s), 0) > 0:
                cnt += 1
    demand_gap[s] = cnt

# 외부 충원 필요 스킬 (잉여 보유자 0명인데 수요 있음)
external_needed = [s for s in ALL_SKILLS
                   if holders_surplus.get(s, 0) == 0 and demand_gap[s] > 0]

# KSS raw 계산
kss_raw_vals = {}
for s in ALL_SKILLS:
    h = holders_surplus.get(s, 0)
    d = demand_gap[s]
    if h == 0 and d > 0:
        kss_raw_vals[s] = np.nan      # 외부 충원 필요 → 정규화 후 최대값으로
    elif h == 0 and d == 0:
        kss_raw_vals[s] = 0.0         # 수요도 없음 → 0
    else:
        kss_raw_vals[s] = d / h

kss_raw = pd.Series(kss_raw_vals)
# nan 제외하고 정규화한 뒤, nan 자리에 1.0 대입
kss_finite = kss_raw.dropna()
if len(kss_finite) > 0 and kss_finite.max() > kss_finite.min():
    kss_norm_finite = (kss_finite - kss_finite.min()) / (kss_finite.max() - kss_finite.min())
else:
    kss_norm_finite = kss_finite * 0
kss_norm = kss_raw.copy()
kss_norm[kss_norm.notna()] = kss_norm_finite
kss_norm[kss_norm.isna()] = 1.0    # 외부 충원 필요 → KSS 최대(1.0)
kss_norm = kss_norm.round(4)

df_kss = pd.DataFrame({
    '스킬': ALL_SKILLS,
    '미달과제수': [demand_gap[s] for s in ALL_SKILLS],
    '잉여보유자': holders_surplus.reindex(ALL_SKILLS).fillna(0).astype(int),
    'KSS_norm': kss_norm.reindex(ALL_SKILLS),
    '외부충원필요': [s in external_needed for s in ALL_SKILLS]
}).sort_values('KSS_norm', ascending=False).reset_index(drop=True)

df_kss['병목'] = (df_kss['잉여보유자'] < df_kss['미달과제수']) & (~df_kss['외부충원필요'])

# ── 스킬 난이도 계산 (전체 기준) ─────────────────────────────────────────
difficulty_raw = pd.Series({
    s: (1 - holders_lv4.get(s,0)/holders_all.get(s,1)) if holders_all.get(s,0)>0 else 0.0
    for s in ALL_SKILLS})

df_diff = pd.DataFrame({
    '스킬': ALL_SKILLS,
    '전체보유자': holders_all.reindex(ALL_SKILLS).fillna(0).astype(int),
    '레벨4이상': holders_lv4.reindex(ALL_SKILLS).fillna(0).astype(int),
    '난이도': difficulty_raw
}).sort_values('난이도', ascending=False).reset_index(drop=True)

print('IDF / KSS / 난이도 계산 완료')
print(f"희귀({df_idf[df_idf['분류']=='희귀'].shape[0]}개) | "
      f"보통({df_idf[df_idf['분류']=='보통'].shape[0]}개) | "
      f"보편({df_idf[df_idf['분류']=='보편'].shape[0]}개)")
print(f"KSS 미달 과제 있는 스킬: {(df_kss['미달과제수']>0).sum()}개 | "
      f"병목 스킬: {df_kss['병목'].sum()}개")
if external_needed:
    print(f"⚠️  외부 충원 필요 스킬 ({len(external_needed)}개): {external_needed}")

## EDA — 카드 1 : 스킬 희귀도 (IDF, 전체 기준)

In [ ]:
TOP_N = 15
top_rare   = df_idf.head(TOP_N)
top_common = df_idf.tail(TOP_N).iloc[::-1]
color_map  = {'희귀':'#E05C5C', '보통':'#7A9CC6', '보편':'#6DBF8A'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('카드 1 : 스킬 희귀도 (IDF_norm, 전체 500명 기준)', fontsize=13, fontweight='bold', y=1.01)

for ax, df_sub, title in [
    (axes[0], top_rare,   f'희귀 스킬 TOP {TOP_N}'),
    (axes[1], top_common, f'보편 스킬 TOP {TOP_N}'),
]:
    colors = [color_map[c] for c in df_sub['분류']]
    bars = ax.barh(df_sub['스킬'], df_sub['IDF_norm'], color=colors)
    for bar, h in zip(bars, df_sub['전체보유자']):
        ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2, f'{h}명', va='center', fontsize=8)
    ax.set_xlim(0, 1.15); ax.set_xlabel('IDF_norm'); ax.set_title(title, fontsize=11); ax.invert_yaxis()

patches = [mpatches.Patch(color=v, label=k) for k,v in color_map.items()]
fig.legend(handles=patches, loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.05))
plt.tight_layout(); plt.show()

### 카드 1 선택 — 희귀도를 적합도에 반영할 스킬

In [ ]:
cb_idf = {}; rows_idf = []
for _, row in df_idf.iterrows():
    s = row['스킬']
    label = f"{s}  │  IDF={row['IDF_norm']:.3f}  │  전체보유자={row['전체보유자']}명  │  [{row['분류']}]"
    cb = widgets.Checkbox(value=False, description=label,
                          layout=widgets.Layout(width='560px'), style={'description_width':'initial'})
    cb_idf[s] = cb; rows_idf.append(cb)

btn_rare  = widgets.Button(description='희귀 전체 선택', button_style='danger',  layout=widgets.Layout(width='140px'))
btn_clr1  = widgets.Button(description='전체 해제',      button_style='',        layout=widgets.Layout(width='120px'))
def sel_rare(_):
    for s,cb in cb_idf.items():
        cb.value = df_idf[df_idf['스킬']==s]['분류'].iloc[0]=='희귀'
def clr1(_):
    for cb in cb_idf.values(): cb.value=False
btn_rare.on_click(sel_rare); btn_clr1.on_click(clr1)
cnt1 = widgets.HTML(''); 
def upd1(change):
    cnt1.value = f'<b>{sum(cb.value for cb in cb_idf.values())}개 선택됨</b>'
for cb in cb_idf.values(): cb.observe(upd1, names='value')

display(widgets.VBox([
    widgets.HTML('<b>희귀도를 적합도에 반영할 스킬 선택 (I_idf=1)</b><br>'),
    widgets.HBox([btn_rare, btn_clr1, cnt1]),
    widgets.VBox(rows_idf, layout=widgets.Layout(height='300px', overflow_y='scroll',
                                                   border='1px solid #ccc', padding='6px'))]))

## EDA — 카드 2 : 수요-공급 불균형 (KSS, 문제 2 전용)

In [ ]:
TOP_N = 15
top_kss = df_kss.head(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('카드 2 : 수요-공급 불균형\n(수요=기존팀원 미달 과제 수 / 공급=잉여 인력 보유자 수)',
             fontsize=12, fontweight='bold', y=1.02)

def kss_color(row):
    if row['외부충원필요']: return '#C0392Bcc'
    if row['병목']:         return '#E67E22cc'
    return '#5D8AA8cc'

bar_colors = [kss_color(row) for _, row in top_kss.iterrows()]
bars = axes[0].barh(top_kss['스킬'], top_kss['KSS_norm'], color=bar_colors)
for bar, row in zip(bars, top_kss.itertuples()):
    label = ('외부충원필요' if row.외부충원필요
             else f'미달{row.미달과제수}/공급{row.잉여보유자}명')
    axes[0].text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                 label, va='center', fontsize=8)
axes[0].set_xlim(0, 1.3); axes[0].set_xlabel('KSS_norm'); axes[0].invert_yaxis()
axes[0].set_title(f'수요집중도 TOP {TOP_N}', fontsize=11)
ext  = mpatches.Patch(color='#C0392Bcc', label='외부충원필요')
btl  = mpatches.Patch(color='#E67E22cc', label='병목')
nor  = mpatches.Patch(color='#5D8AA8cc', label='일반')
axes[0].legend(handles=[ext, btl, nor], fontsize=9)

colors_sc = [('#C0392Bcc' if r['외부충원필요'] else
              '#E67E22cc' if r['병목'] else '#95A5A6cc')
             for _, r in df_kss.iterrows()]
axes[1].scatter(df_kss['잉여보유자'], df_kss['미달과제수'], c=colors_sc, alpha=0.7, s=60)
mx = max(df_kss['잉여보유자'].max(), df_kss['미달과제수'].max())
axes[1].plot([0,mx],[0,mx],'k--',alpha=0.4,label='수요=공급 기준선')
for _, r in df_kss[df_kss['외부충원필요']].iterrows():
    axes[1].annotate(r['스킬'],(r['잉여보유자'],r['미달과제수']),fontsize=7,
                     xytext=(4,4),textcoords='offset points',color='#C0392B')
for _, r in df_kss[df_kss['병목']].iterrows():
    axes[1].annotate(r['스킬'],(r['잉여보유자'],r['미달과제수']),fontsize=7,
                     xytext=(4,4),textcoords='offset points',color='#E67E22')
axes[1].set_xlabel('잉여 인력 보유자 수 (공급)')
axes[1].set_ylabel('기존팀원 미달 과제 수 (수요)')
axes[1].set_title('수요 vs 공급 분포', fontsize=11); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

if external_needed:
    print(f"⚠️  외부 충원 필요 스킬 ({len(external_needed)}개): {external_needed}")
    print("   잉여 인력 중 해당 스킬 보유자가 없어 재배치로는 채울 수 없습니다.")
print(f"병목 스킬 {df_kss['병목'].sum()}개 | "
      f"수요 있는 스킬 {(df_kss['미달과제수']>0).sum()}개")

### 카드 2 선택 — 수요집중도를 적합도에 반영할 스킬

In [ ]:
cb_kss = {}; rows_kss = []
for _, row in df_kss.iterrows():
    s = row['스킬']
    if row['외부충원필요']:
        flag = '🔴외부충원필요'
    elif row['병목']:
        flag = '🟠병목'
    else:
        flag = ''
    label = (f"{s}  │  KSS={row['KSS_norm']:.3f}  │  "
             f"미달과제{row['미달과제수']}/잉여보유{row['잉여보유자']}명  {flag}")
    cb = widgets.Checkbox(value=False, description=label,
                          layout=widgets.Layout(width='620px'), style={'description_width':'initial'})
    cb_kss[s] = cb; rows_kss.append(cb)

btn_btl  = widgets.Button(description='병목 전체 선택',    button_style='warning', layout=widgets.Layout(width='140px'))
btn_ext  = widgets.Button(description='외부충원 전체 선택', button_style='danger',  layout=widgets.Layout(width='160px'))
btn_clr2 = widgets.Button(description='전체 해제',         button_style='',        layout=widgets.Layout(width='120px'))

def sel_btl(_):
    for s,cb in cb_kss.items():
        row = df_kss[df_kss['스킬']==s].iloc[0]
        cb.value = bool(row['병목'])
def sel_ext(_):
    for s,cb in cb_kss.items():
        row = df_kss[df_kss['스킬']==s].iloc[0]
        cb.value = bool(row['외부충원필요'])
def clr2(_):
    for cb in cb_kss.values(): cb.value=False

btn_btl.on_click(sel_btl); btn_ext.on_click(sel_ext); btn_clr2.on_click(clr2)
cnt2 = widgets.HTML('')
def upd2(change): cnt2.value = f'<b>{sum(cb.value for cb in cb_kss.values())}개 선택됨</b>'
for cb in cb_kss.values(): cb.observe(upd2, names='value')

display(widgets.VBox([
    widgets.HTML('<b>수요집중도를 적합도에 반영할 스킬 선택 (I_kss=1)</b><br>'
                 '<small>🔴 외부충원필요: 잉여 인력 중 보유자 없음 (적합도 영향 없음, 참고용)</small><br>'),
    widgets.HBox([btn_btl, btn_ext, btn_clr2, cnt2]),
    widgets.VBox(rows_kss, layout=widgets.Layout(height='300px', overflow_y='scroll',
                                                   border='1px solid #ccc', padding='6px'))]))

## EDA — 카드 3 : 스킬 난이도 (전체 기준)

In [ ]:
TOP_N = 15
top_hard = df_diff.head(TOP_N)
top_easy = df_diff.tail(TOP_N).iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('카드 3 : 스킬 난이도 (전체 기준)', fontsize=13, fontweight='bold', y=1.01)

for ax, df_sub, title, col in [
    (axes[0], top_hard, f'어려운 스킬 TOP {TOP_N}', '#8E44AD'),
    (axes[1], top_easy, f'쉬운 스킬 TOP {TOP_N}',   '#27AE60'),
]:
    bars = ax.barh(df_sub['스킬'], df_sub['난이도'], color=col, alpha=0.75)
    for bar, row in zip(bars, df_sub.itertuples()):
        ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                f'lv4+:{row.레벨4이상}/{row.전체보유자}명', va='center', fontsize=8)
    ax.set_xlim(0, 1.2); ax.set_xlabel('난이도'); ax.set_title(title, fontsize=11); ax.invert_yaxis()

plt.tight_layout(); plt.show()

### 카드 3 선택 — 과제 난이도 계산에 포함할 스킬

In [ ]:
cb_diff = {}; rows_diff = []
for _, row in df_diff.iterrows():
    s = row['스킬']
    label = f"{s}  │  난이도={row['난이도']:.3f}  │  lv4+:{row['레벨4이상']}/{row['전체보유자']}명"
    cb = widgets.Checkbox(value=False, description=label,
                          layout=widgets.Layout(width='520px'), style={'description_width':'initial'})
    cb_diff[s] = cb; rows_diff.append(cb)

btn_top20 = widgets.Button(description='어려운 스킬 TOP20', button_style='warning', layout=widgets.Layout(width='160px'))
btn_clr3  = widgets.Button(description='전체 해제',          button_style='',        layout=widgets.Layout(width='120px'))
def sel_top20(_):
    top20 = set(df_diff.head(20)['스킬'])
    for s,cb in cb_diff.items(): cb.value = s in top20
def clr3(_):
    for cb in cb_diff.values(): cb.value=False
btn_top20.on_click(sel_top20); btn_clr3.on_click(clr3)
cnt3 = widgets.HTML('')
def upd3(change): cnt3.value = f'<b>{sum(cb.value for cb in cb_diff.values())}개 선택됨</b>'
for cb in cb_diff.values(): cb.observe(upd3, names='value')

display(widgets.VBox([
    widgets.HTML('<b>과제 난이도 계산에 포함할 스킬 선택</b><br>'),
    widgets.HBox([btn_top20, btn_clr3, cnt3]),
    widgets.VBox(rows_diff, layout=widgets.Layout(height='300px', overflow_y='scroll',
                                                   border='1px solid #ccc', padding='6px'))]))

## EDA — 카드 4 : 인재 유형 분류 (잉여 인력 기준, 인사이트 전용)

In [ ]:
talent_rows = []
for emp_id, row in surplus_wide.iterrows():
    skills_held = row[row > 0]
    n = len(skills_held)
    avg = skills_held.mean() if n>0 else 0
    std = skills_held.std()  if n>1 else 0
    talent_rows.append({'사번':emp_id, '보유스킬수':n, '평균레벨':avg, '레벨분산':std})

df_talent = pd.DataFrame(talent_rows)
med_n   = df_talent['보유스킬수'].median()
med_std = df_talent['레벨분산'].median()

def classify_talent(r):
    if r['보유스킬수'] <= med_n and r['레벨분산'] >= med_std: return '전문가형'
    elif r['보유스킬수'] > med_n and r['레벨분산'] < med_std: return '제너럴리스트형'
    else: return '혼합형'
df_talent['유형'] = df_talent.apply(classify_talent, axis=1)

type_colors = {'전문가형':'#2E86AB', '제너럴리스트형':'#E84855', '혼합형':'#F4A261'}
colors_sc   = [type_colors[t] for t in df_talent['유형']]
type_counts = df_talent['유형'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('카드 4 : 인재 유형 분류 (잉여 인력 기준)', fontsize=13, fontweight='bold')

axes[0].scatter(df_talent['보유스킬수'], df_talent['레벨분산'], c=colors_sc, alpha=0.6, s=50)
axes[0].axvline(med_n,   color='gray', linestyle='--', alpha=0.6, label=f'스킬수 중앙값({med_n:.0f})')
axes[0].axhline(med_std, color='gray', linestyle=':',  alpha=0.6, label=f'분산 중앙값({med_std:.2f})')
for t,c in type_colors.items(): axes[0].scatter([],[],c=c,label=t,s=60)
axes[0].set_xlabel('보유 스킬 수'); axes[0].set_ylabel('레벨 표준편차')
axes[0].set_title('보유 스킬 수 vs 레벨 분산'); axes[0].legend(fontsize=9)

axes[1].pie(type_counts.values,
            labels=[f"{k}\n({v}명, {v/N_SURPLUS*100:.1f}%)" for k,v in type_counts.items()],
            colors=[type_colors[k] for k in type_counts.index], startangle=90)
axes[1].set_title('잉여 인력 유형 분포')

plt.tight_layout(); plt.show()
print('\n[유형별 평균 스킬 현황]')
print(df_talent.groupby('유형')[['보유스킬수','평균레벨','레벨분산']].mean().round(2))

## 적합도 매트릭스 산출
> **카드 1·2·3 선택 완료 후 실행하세요.**

In [ ]:
selected_idf  = [s for s,cb in cb_idf.items()  if cb.value]
selected_kss  = [s for s,cb in cb_kss.items()  if cb.value]
selected_diff = [s for s,cb in cb_diff.items() if cb.value]

print(f'카드 1 선택: {len(selected_idf)}개 | 카드 2 선택: {len(selected_kss)}개 | 카드 3 선택: {len(selected_diff)}개')

# 과제 난이도 (후보 과제만)
proj_difficulty = {}
for p in CANDIDATE_P:
    req_s = proj_req.get(p, [])
    sel   = [s for s in selected_diff if s in req_s]
    proj_difficulty[p] = sum(difficulty_raw.get(s,0) for s in sel)
maxd = max(proj_difficulty.values()) if proj_difficulty else 1
proj_diff_norm = {p: v/maxd for p,v in proj_difficulty.items()}

print('\n[후보 과제 난이도 (정규화)]')
diff_df = pd.DataFrame({'과제':list(proj_diff_norm.keys()), '난이도_norm':list(proj_diff_norm.values())})
print(diff_df.sort_values('난이도_norm', ascending=False).to_string(index=False))

In [ ]:
# 스킬 중요도
W3 = 0.5
skill_imp = {}
for s in ALL_SKILLS:
    skill_imp[s] = (1
                    + (1 if s in selected_idf else 0) * idf_norm.get(s, 0)
                    + (1 if s in selected_kss else 0) * kss_norm.get(s, 0))

# 잉여 인력 × 후보 과제 fit_matrix
surplus_ids = surplus_wide.index.tolist()
fit_matrix = pd.DataFrame(index=surplus_ids, columns=CANDIDATE_P, dtype=float)

for p in CANDIDATE_P:
    req_s  = [s for s in proj_req.get(p, []) if s in surplus_wide.columns]
    imp    = np.array([skill_imp[s] for s in req_s])
    levels = surplus_wide[req_s].values
    base   = (levels * imp).sum(axis=1)
    fit_matrix[p] = base * (1 + W3 * proj_diff_norm[p])

print(f'적합도 매트릭스 생성 완료: {N_SURPLUS}명 × {len(CANDIDATE_P)}과제')
stats = fit_matrix.agg(['max','min','mean','std']).T.round(2)
stats['max-min'] = stats['max'] - stats['min']
print(stats.sort_values('max-min', ascending=False).to_string())

## 3단계 — 배치 옵션 + ILP 재배치

In [ ]:
import pulp, time

# ── 옵션 설정 ────────────────────────────────────────────────────────────────
AVG_LEVEL         = 2.8
LAM_AVG           = 50.0
LAM_GENDER        = 2.7
LAM_RANK          = 2.7
ENABLE_AVG_LEVEL  = True
ENABLE_GENDER     = True
ENABLE_RANK       = True
ENABLE_SKILL_COV  = True    # 보유 커버리지 하드제약
HOLD_LEVEL        = 1       # 보유 인정 최소 레벨
MIN_HOLDERS       = 1       # 과제당 스킬별 최소 보유 인원
PRIORITY          = {}      # 과제 중요도 (없으면 전부 1.0)
TIME_LIMIT        = 300
GAP               = 0.01

rmap   = {'사원':1,'대리':2,'과장':3,'차장':4,'부장':5}
fem_s  = (surplus_hr['성별']=='여').values.astype(float)
rk_s   = surplus_hr['직위'].map(rmap).fillna(3).values.astype(float)
pool_f = (pd.concat([surplus_hr, existing_hr])['성별']=='여').mean()
pool_r = pd.concat([surplus_hr, existing_hr])['직위'].map(rmap).fillna(3).mean()
pri    = {p: PRIORITY.get(p, 1.0) for p in CANDIDATE_P}

n_s = N_SURPLUS
n_p = len(CANDIDATE_P)
P_idx = {p:j for j,p in enumerate(CANDIDATE_P)}

print('배치 옵션 확인')
print(f'  보유 커버리지(하드): {ENABLE_SKILL_COV} | 보유레벨 {HOLD_LEVEL} | 최소 {MIN_HOLDERS}명')
print(f'  평균 레벨 기준(소프트): {ENABLE_AVG_LEVEL} | 기준 {AVG_LEVEL}')
print(f'  성별/직급 균형: {ENABLE_GENDER}/{ENABLE_RANK}')
print(f'  과제별 추가 상한: {MAX_ADD}')

### 보유 커버리지 사전 검사

In [ ]:
infeasible = []
if ENABLE_SKILL_COV:
    for p in CANDIDATE_P:
        for s in proj_req.get(p, []):
            # 기존팀원이 이미 충족하면 검사 불필요
            if team_skill_holders.get((p,s), 0) >= MIN_HOLDERS:
                continue
            # 잉여 인력 중 보유자 수 확인
            n_hold = int((surplus_wide[s] >= HOLD_LEVEL).sum()) if s in surplus_wide.columns else 0
            if n_hold < MIN_HOLDERS:
                infeasible.append((p, s, n_hold))

if infeasible:
    print('⚠️  보유 커버리지 하드제약 충족 불가능:')
    for p,s,h in infeasible:
        print(f'   - {p} / {s}: 잉여 보유자 {h}명 (기존팀원도 미보유)')
    print('\n→ ENABLE_SKILL_COV를 False로 끄거나 HOLD_LEVEL을 낮추세요.')
else:
    print('✅ 보유 커버리지 사전 검사 통과')

### ILP 모델 구성

In [ ]:
t0 = time.time()
prob = pulp.LpProblem('reassign', pulp.LpMaximize)
F = fit_matrix.values  # (n_s × n_p)

x = {(i,j): pulp.LpVariable(f'x_{i}_{j}', cat='Binary')
     for i in range(n_s) for j in range(n_p)}

obj = pulp.lpSum(F[i,j] * pri[CANDIDATE_P[j]] * x[(i,j)]
                 for i in range(n_s) for j in range(n_p))

# ── 하드 ① 잉여 인력 1인 최대 1과제 (== 1 → <= 1 수정) ───────────────────
# 상한 합계 < 잉여 인원이면 모두 배치 불가 → <= 1로 변경, 미배치 페널티 추가
UNASSIGNED_PENALTY = 100   # 미배치 1명당 감점
for i in range(n_s):
    prob += pulp.lpSum(x[(i,j)] for j in range(n_p)) <= 1
    obj -= UNASSIGNED_PENALTY * (1 - pulp.lpSum(x[(i,j)] for j in range(n_p)))

# ── 하드 ② 과제별 추가 인원 ≤ 상한 ────────────────────────────────────────
for j, p in enumerate(CANDIDATE_P):
    prob += pulp.lpSum(x[(i,j)] for i in range(n_s)) <= MAX_ADD[p]

# ── 하드 ③ 보유 커버리지 ────────────────────────────────────────────────────
if ENABLE_SKILL_COV:
    for j, p in enumerate(CANDIDATE_P):
        for s in proj_req.get(p, []):
            if team_skill_holders.get((p,s), 0) >= MIN_HOLDERS:
                continue
            holds = [(1 if s in surplus_wide.columns and surplus_wide.iloc[i][s] >= HOLD_LEVEL else 0)
                     for i in range(n_s)]
            if sum(holds) > 0:
                prob += pulp.lpSum(holds[i]*x[(i,j)] for i in range(n_s)) >= MIN_HOLDERS

# ── 소프트 ① 평균 레벨 기준 ──────────────────────────────────────────────────
# 2번 수정: 분모를 MAX_ADD 고정값이 아닌 실제 배치 인원으로
# AVG_LEVEL × (Nj + Σx) = AVG_LEVEL×Nj + AVG_LEVEL×Σx (선형 유지)
avg_slack = {}
if ENABLE_AVG_LEVEL:
    for j, p in enumerate(CANDIDATE_P):
        for s in proj_req.get(p, []):
            sl = pulp.LpVariable(f'avg_{j}_{s}', lowBound=0)
            avg_slack[(j,s)] = sl
            fixed = team_skill_sum.get((p,s), 0)
            actual_lv = pulp.lpSum(
                (float(surplus_wide.iloc[i][s]) if s in surplus_wide.columns else 0) * x[(i,j)]
                for i in range(n_s))
            actual_add = pulp.lpSum(x[(i,j)] for i in range(n_s))
            prob += (fixed + actual_lv + sl
                     >= AVG_LEVEL * Nj_existing[p] + AVG_LEVEL * actual_add)
            obj -= LAM_AVG * sl

# ── 소프트 ② 성별 균형 ──────────────────────────────────────────────────────
# 3번 수정: 분모를 실제 배치 인원으로
# pool_f × (Nj + Σx) = pool_f×Nj + Σ(pool_f×x) (선형 유지)
if ENABLE_GENDER:
    for j, p in enumerate(CANDIDATE_P):
        dp = pulp.LpVariable(f'gdp_{j}', lowBound=0)
        dm = pulp.LpVariable(f'gdm_{j}', lowBound=0)
        fixed_f = float((existing_hr[existing_hr['소속과제명']==p]['성별']=='여').sum())
        prob += (fixed_f
                 + pulp.lpSum(fem_s[i]*x[(i,j)] for i in range(n_s))
                 - pool_f * Nj_existing[p]
                 - pulp.lpSum(pool_f * x[(i,j)] for i in range(n_s))
                 == dp - dm)
        obj -= LAM_GENDER * (dp + dm)

# ── 소프트 ③ 직급 균형 ──────────────────────────────────────────────────────
# 3번 수정: 동일하게 실제 배치 인원으로
if ENABLE_RANK:
    for j, p in enumerate(CANDIDATE_P):
        dp = pulp.LpVariable(f'rdp_{j}', lowBound=0)
        dm = pulp.LpVariable(f'rdm_{j}', lowBound=0)
        fixed_r = float(existing_hr[existing_hr['소속과제명']==p]['직위'].map(rmap).fillna(3).sum())
        prob += (fixed_r
                 + pulp.lpSum(rk_s[i]*x[(i,j)] for i in range(n_s))
                 - pool_r * Nj_existing[p]
                 - pulp.lpSum(pool_r * x[(i,j)] for i in range(n_s))
                 == dp - dm)
        obj -= LAM_RANK * (dp + dm)

prob += obj
build_t = time.time() - t0
print(f'모델 구성 완료 | {build_t:.1f}s | 변수 {len(prob.variables()):,} | 제약 {len(prob.constraints):,}')

# 미배치 가능 여부 사전 안내
max_capacity = sum(MAX_ADD.values())
if max_capacity < n_s:
    print(f"\n⚠️  총 수용 가능 인원({max_capacity}명) < 잉여 인원({n_s}명)")
    print(f"   최소 {n_s - max_capacity}명이 미배치될 수 있습니다.")
    print("   과제별 상한을 늘리거나 후보 과제를 추가하세요.")
else:
    print(f"\n✅ 총 수용 가능 인원({max_capacity}명) ≥ 잉여 인원({n_s}명) — 전원 배치 가능")

### 풀이

In [ ]:
solver, sname = None, ""
for _try in ["HiGHS_API", "HiGHS_CMD", "CBC"]:
    try:
        if _try == "HiGHS_API":
            cand = pulp.HiGHS(msg=True, timeLimit=TIME_LIMIT, gapRel=GAP); cname = "HiGHS(API)"
        elif _try == "HiGHS_CMD":
            cand = pulp.HiGHS_CMD(msg=True, timeLimit=TIME_LIMIT, gapRel=GAP); cname = "HiGHS(CMD)"
        else:
            cand = pulp.PULP_CBC_CMD(msg=True, timeLimit=TIME_LIMIT, gapRel=GAP); cname = "CBC"
        if cand.available():
            solver, sname = cand, cname; break
    except Exception:
        continue
if solver is None:
    solver = pulp.PULP_CBC_CMD(msg=True, timeLimit=TIME_LIMIT, gapRel=GAP); sname = "CBC"

print(f'선택된 솔버: {sname}')
t1 = time.time()
prob.solve(solver)
solve_t = time.time() - t1
print(f'\n솔버 {sname} | 상태 {pulp.LpStatus[prob.status]} | 풀이 {solve_t:.1f}s')

if pulp.LpStatus[prob.status] == 'Infeasible':
    print('\n❌ 해가 없습니다. 보유 커버리지 또는 추가 상한이 너무 엄격할 수 있습니다.')
    print('   사전 검사 결과 확인 후 옵션을 완화하세요.')

## 결과 요약 & 저장

In [ ]:
def _asg(i):
    for j in range(n_p):
        v = x[(i,j)].value()
        if v and v > 0.5: return j
    return -1   # -1 = 미배치

asg = {i: _asg(i) for i in range(n_s)}

# 미배치 / 배치 분리
unassigned_idx = [i for i in range(n_s) if asg[i] == -1]
assigned_idx   = [i for i in range(n_s) if asg[i] != -1]
opt_total = sum(F[i, asg[i]] for i in assigned_idx)
unmet = sum(1 for v in avg_slack.values() if v.value() and v.value() > 1e-6) if avg_slack else 0

print('=' * 60)
print(f'솔버 {sname} | 상태 {pulp.LpStatus[prob.status]}')
print(f'구성 {build_t:.1f}s | 풀이 {solve_t:.1f}s')
print(f'배치 완료: {len(assigned_idx)}명 | 미배치: {len(unassigned_idx)}명')
print(f'배치 인원 적합도 총합: {opt_total:.1f}')
if ENABLE_AVG_LEVEL:
    print(f'평균 레벨({AVG_LEVEL}) 미달: {unmet}/{len(avg_slack)}건')
if ENABLE_SKILL_COV:
    print(f'보유 커버리지(하드): 충족')
print('=' * 60)

# 미배치 인원 표시
if unassigned_idx:
    print(f"\n⚠️  미배치 인원 ({len(unassigned_idx)}명):")
    for i in unassigned_idx:
        row = surplus_hr.iloc[i]
        name = str(row['Last Name']) + str(row['First Name'])
        print(f"   - {row['사번']} {name} ({row['직위']}, {row['연차']}년차)")
    print("   → 과제별 상한을 늘리거나 후보 과제를 추가하세요.")

# 결과 저장 (배치된 인원만)
out = surplus_hr.copy()
out['재배치과제'] = [CANDIDATE_P[asg[i]] if asg[i] != -1 else '미배치' for i in range(n_s)]
out['적합도']    = [round(F[i, asg[i]], 2) if asg[i] != -1 else 0 for i in range(n_s)]
out.to_excel('재배치결과.xlsx', index=False)
print('\n→ 재배치결과.xlsx 저장 완료')

## 팀별 재배치 전후 비교

In [ ]:
summary = []
for j, p in enumerate(CANDIDATE_P):
    new_members_idx = [i for i in range(n_s) if asg[i]==j]
    n_add   = len(new_members_idx)
    n_total = Nj_existing[p] + n_add

    # 추가된 인원 적합도 평균
    avg_fit_add = (sum(F[i, j] for i in new_members_idx) / n_add) if n_add > 0 else 0

    # 성비
    n_f_exist  = (existing_hr[existing_hr['소속과제명']==p]['성별']=='여').sum()
    n_f_add    = sum(fem_s[i] for i in new_members_idx)
    fem_before = round(n_f_exist / Nj_existing[p] * 100, 1) if Nj_existing[p] > 0 else 0
    fem_after  = round((n_f_exist + n_f_add) / n_total * 100, 1) if n_total > 0 else 0

    # 직급
    rk_exist = existing_hr[existing_hr['소속과제명']==p]['직위'].map(rmap).fillna(3)
    rk_add   = sum(rk_s[i] for i in new_members_idx)
    avg_rk_before = round(rk_exist.mean(), 2) if len(rk_exist) > 0 else 0
    avg_rk_after  = round((rk_exist.sum() + rk_add) / n_total, 2) if n_total > 0 else 0

    # 추가된 인원 사번/이름
    added_ids   = [surplus_hr.iloc[i]['사번'] for i in new_members_idx]
    added_names = [str(surplus_hr.iloc[i]['Last Name']) + str(surplus_hr.iloc[i]['First Name'])
                   for i in new_members_idx]

    summary.append({
        '과제': p,
        '기존인원': Nj_existing[p],
        '추가인원': n_add,
        '최종인원': n_total,
        '추가인원 평균적합도': round(avg_fit_add, 2),
        '여성비율 전(%)': fem_before,
        '여성비율 후(%)': fem_after,
        '평균직급 전': avg_rk_before,
        '평균직급 후': avg_rk_after,
        '추가된 인원': ', '.join(added_names) if added_names else '없음'
    })

summary_df = pd.DataFrame(summary)
print('[팀별 재배치 결과 요약]')
display(summary_df.drop(columns=['추가된 인원']))

print('\n[과제별 추가 인원 상세]')
for _, row in summary_df[summary_df['추가인원'] > 0].iterrows():
    print(f"  {row['과제']} ({row['추가인원']}명): {row['추가된 인원']}")
if summary_df['추가인원'].sum() < n_s:
    print(f"\n⚠️  미배치 인원: {n_s - int(summary_df['추가인원'].sum())}명")
    print("   추가 상한을 늘리거나 후보 과제를 추가하세요.")